# KernelForge on a T4 (Colab or Kaggle)

Builds five hand-written CUDA kernels, gates on correctness against a NumPy
reference, sweeps the benchmark, and profiles the two scoring kernels to show why
v4 is faster than v3. Run top to bottom; the whole thing takes about 15 minutes,
most of it the 1M-row sweep.

**First: attach a T4.** Colab: Runtime > Change runtime type > T4 GPU. Kaggle:
Settings > Accelerator > GPU T4, with Internet on for the clone.

Nothing counts until cell 4 reports **0 skipped** and cell 5 writes real rows to
`bench/results.csv`. Skipped tests are what a missing GPU looks like, and a screen
of `s` reads as success while proving nothing.


## 0. Environment, asserted before anything is installed


In [ ]:
import subprocess, sys, shutil
from pathlib import Path

# Works on Kaggle (/kaggle/working) and Colab (/content).
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
REPO = BASE / 'kernelforge-cuda'

!nvidia-smi

def torch_probe(expr):
    """Read torch in a subprocess so this kernel never imports it."""
    p = subprocess.run([sys.executable, '-c', f'import torch; print({expr})'],
                       capture_output=True, text=True)
    return p.stdout.strip() or f'MISSING ({p.stderr.strip()[:80]})'

before = torch_probe('torch.__version__')
print('torch before install:', before)
assert '+cpu' not in before, (
    'this session already has a CPU-only torch. Stop the session entirely and start a '
    'new one -- restarting the kernel will not replace the wheel on disk.'
)

# nvcc has to exist, and it has to be the toolkit that matches this driver. A build
# from a mismatched toolkit links and loads fine, then fails at kernel launch.
assert shutil.which('nvcc'), 'no nvcc on PATH: this runtime has no CUDA toolkit'
!nvcc --version | tail -2
print('torch was built against CUDA:', torch_probe('torch.version.cuda'))


## 1. Detect the arch from the device, never hardcode it

`-arch` is the one build flag that fails *late*: build for the wrong compute
capability and the library still loads, ctypes still binds every symbol, and the
first launch returns error 209, `no kernel image is available`. T4 is `sm_75`,
A100 `sm_80`, L4 `sm_89`.


In [ ]:
cap = subprocess.run([sys.executable, '-c',
    'import torch; c=torch.cuda.get_device_capability(); print(f"{c[0]}{c[1]} {torch.cuda.get_device_name(0)}")'],
    capture_output=True, text=True).stdout.strip()
assert cap, 'no CUDA device visible to torch -- attach a GPU runtime and rerun'
num, name = cap.split(' ', 1)
ARCH = f'sm_{num}'
print(f'device: {name}  ->  building for {ARCH}')
if ARCH != 'sm_75':
    print('NOTE: every number in RESULTS.md is T4 (sm_75). Record this device in the',
          'results table -- two GPUs in one table is not a benchmark.')


## 2. Clone and install

Two rules, both learned the hard way in this portfolio.

1. **Never install torch.** Colab and Kaggle ship one built against their driver.
   Letting pip resolve it swaps in a CPU wheel and silently removes the GPU — which
   here would not error, it would just skip every GPU test.
2. **`--no-deps`, and pin the rest.** Anything pip pulls in unpinned is a variable
   in a measurement that is supposed to have one.


In [ ]:
BRANCH = 'main'
if not (REPO / '.git').exists():
    !git clone --depth 1 --branch {BRANCH} https://github.com/mghadia1/kernelforge-cuda.git {REPO}
assert (REPO / '.git').exists(), 'clone failed; check Internet is enabled (Kaggle: Settings > Internet)'
%cd {REPO}
!git log --oneline -1

!python -m pip install -q -e . --no-deps
!python -m pip install -q --no-deps 'pytest==8.3.4' 'tabulate==0.9.0'

after = torch_probe('torch.__version__')
print('torch after install :', after)
assert after == before, (
    f'the install replaced torch ({before} -> {after}). Stop the session, start a new '
    'one, and report this so the pins can be fixed.'
)


## 3. Build

One shared library holds v0-v4 and the cuBLAS baseline, so the benchmark switches
between them by symbol name through ctypes.


In [ ]:
!make ARCH={ARCH} 2>&1 | tail -20

import sys; sys.path.insert(0, 'src')
import runner, reference
assert runner.available(), f'library did not load or no device: {runner.load_error()!r}'
print('loaded:', runner.library_path())
print('device seen by the kernels:', runner.device_name())
print('implementations:', runner.ALL_IMPLS)

# The arch check from cell 1, actually exercised: this launches every kernel, so a
# wrong -arch fails here with CUDA error 209 instead of halfway through the sweep.
q, X = reference.make_data(1024, 2, seed=0)
for v in runner.ALL_IMPLS:
    runner.run(v, q, X, 5)
print('all implementations launched cleanly')


## 4. Correctness gate

Every kernel must match the NumPy reference within 1e-4 **and** return the same
indices. The cell asserts on the skip count, because skipping is the failure mode
that looks like a pass.

Two of the versions have selection logic whose answer depends on how work is split
across blocks (v3's per-block lists, v4's warp-per-query masked max-reduction).
Both are also simulated on the host, so those tests pass on any machine — the GPU
tests are the ones that can only run here.


In [ ]:
import re
out = subprocess.run([sys.executable, '-m', 'pytest', '-v'],
                     capture_output=True, text=True).stdout
print(out[-3000:])
summary = out.strip().splitlines()[-1]
skipped = int((re.search(r'(\d+) skipped', summary) or [0, 0])[1])
assert 'failed' not in summary and 'error' not in summary, summary
assert skipped == 0, f'{skipped} tests skipped -- the GPU path did not run: {summary}'
print('\nGATE PASSED:', summary)


## 5. Benchmark sweep

15 timed repeats, first discarded, 3 warmup, median and p95, every implementation
verified against the reference before it is timed. Takes a few minutes.


In [ ]:
!python bench/run.py --out bench/results.csv --repeats 15 2>&1 | tail -30


## 6. The tables, generated not retyped

Paste the printed markdown straight into `bench/RESULTS.md`. Retyping numbers by
hand is how a benchmark quietly turns into fiction.


In [ ]:
!pip install -q tabulate
import pandas as pd
df = pd.read_csv('bench/results.csv')
assert bool(df.indices_match.all()), 'an implementation disagreed with the reference -- do not report these timings'
print('device:', df.device.iloc[0], '| worst abs err:', df.max_abs_err.max())

ORDER = ['cpu_numpy','v0_naive','v1_shared','v2_warp','v3_topk','v4_batch','v5_regblock','cublas','torch_gpu']
for b in sorted(df.B.unique()):
    piv = df[df.B == b].pivot_table(index='N', columns='impl', values='median_ms')
    cols = [c for c in ORDER if c in piv.columns]
    print(f'\n### End-to-end latency, median ms (B = {b})\n')
    print(piv[cols].round(3).to_markdown())
    if 'cpu_numpy' in cols:
        spd = piv[cols].rdiv(piv['cpu_numpy'], axis=0).drop(columns=['cpu_numpy'])
        print(f'\n### Speedup over cpu_numpy (x, B = {b})\n')
        print(spd.round(1).to_markdown())


## 7. Where the time goes: v3 vs v4 vs cuBLAS

This is the batch-tiling argument in one table. v4 differs from v3 in exactly one
way — a block owns eight queries instead of one, so each X element loaded is reused
eight times — and the *kernel* line is where that shows up. Expect roughly 5x on
the kernel and far less end-to-end, because uploading the 1.54 GB corpus dominates
the call.

The `cublas` row is included because it is the baseline v4 is compared against, and
it pays a host-side top-k that v4 does not. Its `hosttopk` figure is the size of
that handicap.


In [ ]:
q, X = reference.make_data(1_000_000, 32, seed=0)
for v in ('v3_topk', 'v4_batch', 'v5_regblock', 'cublas'):
    runner.run(v, q, X, 5)                       # warm up, then measure
    _, _, t = runner.run(v, q, X, 5)
    print(f'{v:<9} h2d {t.h2d_ms:8.2f} | kernel {t.kernel_ms:8.2f} | d2h {t.d2h_ms:6.2f} | '
          f'host top-k {t.host_topk_ms:7.2f} | total {t.total_ms:8.2f} ms')


## 8. Nsight Compute: name the kernel, or the number means nothing

The first profiling attempt in this project used `ncu --set basic` with no kernel
filter, reported 3.80% achieved occupancy, and was measuring the *merge* kernel —
32 blocks over 40 SMs — not the scoring kernel anyone cared about. Always pass
`--kernel-name`.

The three scoring kernels tell the optimization story as a sequence of moved
bottlenecks: v3 pinned against DRAM (~69% DRAM, ~37% SM); v4 cuts DRAM traffic 8x
by reusing each X byte across eight queries, which pushes L1/TEX to ~78% as the
new limiter; v5 register-blocks the query tile to cut that shared-memory traffic
by another 4x. Watch which line is highest in each column — that is the
bottleneck, and each version was chosen by reading it.


In [ ]:
!ncu --kernel-name kf_v3_partial --set full python bench/run.py \\
     --profile-once --profile-impl v3_topk --n 100000 --b 32 > /content/ncu_v3.txt 2>&1
!ncu --kernel-name kf_v4_partial --set full python bench/run.py \\
     --profile-once --profile-impl v4_batch --n 100000 --b 32 > /content/ncu_v4.txt 2>&1
!ncu --kernel-name kf_v5_partial --set full python bench/run.py \\
     --profile-once --profile-impl v5_regblock --n 100000 --b 32 > /content/ncu_v5.txt 2>&1

KEYS = ('DRAM Throughput', 'Memory Throughput', 'Compute (SM) Throughput',
        'L1/TEX Cache Throughput', 'L2 Cache Throughput', 'Duration',
        'Theoretical Occupancy', 'Achieved Occupancy')
for tag in ('v3', 'v4', 'v5'):
    print(f'--- kf_{tag}_partial ---')
    for line in open(f'/content/ncu_{tag}.txt'):
        if any(line.strip().startswith(key) for key in KEYS):
            print('  ' + ' '.join(line.split()))
        elif 'peak performance' in line:
            print('  ' + line.strip())


## 9. Save the evidence


In [ ]:
from google.colab import files
files.download('bench/results.csv')


---

### If you edit the source and rebuild mid-session

`make` will rewrite `build/libkernelforge.so`, but a Python kernel that already
loaded it keeps the **old** library mapped — new symbols come back as
`undefined symbol: kf_...`. Either restart the runtime, or point the loader at a
fresh path:

```python
import os, shutil, importlib
shutil.copy('build/libkernelforge.so', 'build/libkf_new.so')
os.environ['KERNELFORGE_LIB'] = str(REPO / 'build/libkf_new.so')
importlib.reload(runner)
```

### After the run

`resume_eligible` flips to yes only when all five hold:

1. `make` succeeded here;
2. the gate in cell 4 passed with **0 skips**;
3. `bench/results.csv` has real rows and RESULTS.md quotes them;
4. the Nsight numbers are recorded and interpreted — or their absence stated;
5. you can explain one design choice and one failure mode unaided.

Good candidates for #5, all now backed by measurements:

- *why the v1 tile row is padded to 33 floats* — at stride 32 every thread in the
  compute phase hits the same shared-memory bank, a 32-way conflict that cancels
  the tiling;
- *why v4's kernel got 5x faster while end-to-end barely moved* — the corpus
  upload dominates the call;
- *why batch tiling does nothing at B = 1* — with one query there is nothing to
  reuse a loaded byte against, which is why v3 and v4 tie in that column.
